# Empirical Data Extraction Pipeline

This notebook processes the raw downloaded text datasets from Twitter, OpenAssistant, and Reddit.
It acts mathematically to extract transition probabilities $P(S_{t+1} | S_t, A_t)$ and saves them as JSON matrices for the Reinforcement Learning agent to use in the `CustomerSupportEnv`.

In [1]:
import pandas as pd
import numpy as np
import json
import os
import glob
from collections import defaultdict

# Common logical mappings for State and Action approximation
def simple_sentiment(text):
    text = str(text).lower()
    angry_words = ['why', 'wtf', 'hate', 'fix', 'issue', 'problem', 'stuck', 'error', 'awful', 'terrible', 'bad']
    happy_words = ['thanks', 'thank you', 'amazing', 'working', 'solved', 'great', 'appreciate', 'perfect']
    
    if any(w in text for w in angry_words): return 0 # Angry/Frustrated (0)
    if any(w in text for w in happy_words): return 2 # Happy/Resolved (2)
    return 1 # Neutral (1)

def dummy_action_label(text):
    text = str(text).lower()
    if 'sorry' in text or 'apologize' in text or 'mistake' in text: return 2     # Affective Repair (2)
    if 'how can i' in text or 'provide' in text or 'dm us' in text: return 0     # Ask For Info (0)
    if 'email' in text or 'call us' in text or 'human' in text: return 3         # Escalate (3)
    if 'update' in text or 'investigating' in text: return 5                     # Proactive Update (5)
    if 'expected' in text or 'soon' in text: return 6                            # Set Expectation (6)
    if 'let us know' in text or 'feedback' in text or 'help' in text: return 4   # Close w/ Feedback (4)
    return 1 # Provide Solution (1) (Default mostly)

## 1. Twitter Customer Support dataset (twcs.csv)
Parsing 500k conversational turns from massive consumer brands.

In [2]:
TWITTER_DATA_PATH = r"D:\SEM_6\RL\Project\my_local_twitter\twcs\twcs.csv"
OUTPUT_MATRIX_TWITTER = "empirical_transition_matrix_twitter.json"

if os.path.exists(TWITTER_DATA_PATH):
    print("Loading Twitter data... (This may take a minute)")
    df = pd.read_csv(TWITTER_DATA_PATH, nrows=500000)
    transitions_tw = defaultdict(lambda: np.zeros(3))
    
    inbound = df[df['inbound'] == True]
    outbound = df[df['inbound'] == False].dropna(subset=['in_response_to_tweet_id'])
    
    for _, reply in outbound.iterrows():
        parent_tweet = inbound[inbound['tweet_id'] == reply['in_response_to_tweet_id']]
        if parent_tweet.empty: continue
            
        current_state = simple_sentiment(parent_tweet.iloc[0]['text'])
        action_idx = dummy_action_label(reply['text'])
        
        # Basic forward-fill assumption for 2-step thread reconstruction
        next_state = 2 if (action_idx == 1 and current_state == 1) else current_state
        transitions_tw[(current_state, action_idx)][next_state] += 1
            
    matrix_tw = {}
    for state in [0, 1, 2]:
        matrix_tw[state] = {}
        for action in range(7):
            counts = transitions_tw.get((state, action), np.array([0., 0., 0.]))
            total = np.sum(counts)
            probs = (counts / total).tolist() if total > 0 else [1/3, 1/3, 1/3]
            matrix_tw[state][action] = probs
            
    with open(OUTPUT_MATRIX_TWITTER, 'w') as f:
        json.dump(matrix_tw, f, indent=4)
    print(f"SUCCESS! Saved {OUTPUT_MATRIX_TWITTER}")
else:
    print("Path not found:", TWITTER_DATA_PATH)

Loading Twitter data... (This may take a minute)
SUCCESS! Saved empirical_transition_matrix_twitter.json


## 2. Reddit DSTC8 Dialogues
Iterating through raw multi-turn alternating text scripts.

In [3]:
REDDIT_DATA_DIR = r"D:\SEM_6\RL\Project\my_local_dataset\training\training\*.txt"
OUTPUT_MATRIX_REDDIT = "empirical_transition_matrix_reddit.json"

files = glob.glob(REDDIT_DATA_DIR)
if files:
    print(f"Loading {len(files)} Reddit log files...")
    transitions_rd = defaultdict(lambda: np.zeros(3))
    
    for filepath in files:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                turns = json.loads(line).get('turns', [])
                
                for i in range(0, len(turns) - 2, 2):
                    current_state = simple_sentiment(turns[i])
                    action_idx = dummy_action_label(turns[i+1])
                    next_state = simple_sentiment(turns[i+2])
                    transitions_rd[(current_state, action_idx)][next_state] += 1
                        
    matrix_rd = {}
    for state in [0, 1, 2]:
        matrix_rd[state] = {}
        for action in range(7):
            counts = transitions_rd.get((state, action), np.array([0., 0., 0.]))
            total = np.sum(counts)
            probs = (counts / total).tolist() if total > 0 else [1/3, 1/3, 1/3]
            matrix_rd[state][action] = probs
            
    with open(OUTPUT_MATRIX_REDDIT, 'w') as f:
        json.dump(matrix_rd, f, indent=4)
    print(f"SUCCESS! Saved {OUTPUT_MATRIX_REDDIT}")
else:
    print("Path not found:", REDDIT_DATA_DIR)

Loading 907 Reddit log files...
SUCCESS! Saved empirical_transition_matrix_reddit.json


## 3. OpenAssistant (oasst1)
Traversing nested conversational JSONL trees.

In [4]:
OASST_DATA_PATH = r"D:\SEM_6\RL\Project\my_local_oasst1\2023-04-12_oasst_ready.trees.jsonl\2023-04-12_oasst_ready.trees.jsonl"
OUTPUT_MATRIX_OASST = "empirical_transition_matrix_oasst.json"

def extract_paths_from_tree(node, current_path=None):
    if current_path is None: current_path = []
    path = current_path + [node]
    if 'replies' not in node or not node['replies']: return [path]
    
    all_paths = []
    for reply in node['replies']:
        all_paths.extend(extract_paths_from_tree(reply, path))
    return all_paths

if os.path.exists(OASST_DATA_PATH):
    print("Loading OpenAssistant conversational arrays...")
    transitions_oa = defaultdict(lambda: np.zeros(3))
    
    with open(OASST_DATA_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            tree = json.loads(line)
            paths = extract_paths_from_tree(tree['prompt'])
            
            for path in paths:
                for i in range(len(path) - 2):
                    if path[i]['role'] == 'prompter' and path[i+1]['role'] == 'assistant' and path[i+2]['role'] == 'prompter':
                        s_curr = simple_sentiment(path[i]['text'])
                        a_idx = dummy_action_label(path[i+1]['text'])
                        s_next = simple_sentiment(path[i+2]['text'])
                        transitions_oa[(s_curr, a_idx)][s_next] += 1
                        
    matrix_oa = {}
    for state in [0, 1, 2]:
        matrix_oa[state] = {}
        for action in range(7):
            counts = transitions_oa.get((state, action), np.array([0., 0., 0.]))
            total = np.sum(counts)
            probs = (counts / total).tolist() if total > 0 else [1/3, 1/3, 1/3]
            matrix_oa[state][action] = probs
            
    with open(OUTPUT_MATRIX_OASST, 'w') as f:
        json.dump(matrix_oa, f, indent=4)
    print(f"SUCCESS! Saved {OUTPUT_MATRIX_OASST}")
else:
    print("Path not found:", OASST_DATA_PATH)

Loading OpenAssistant conversational arrays...
SUCCESS! Saved empirical_transition_matrix_oasst.json
